In [1]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

d:\Ai Data Engineering\RAG_POC\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
# 1. Load your API Key
load_dotenv()

True

In [3]:
# 2. Setup the Embedding Model 
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2-preview")

In [4]:
# 3. Load the EXISTING Vector Store
persist_directory = "../chroma_db"

print("--- Loading your SQL Database ---")
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)
print(f"Success! Loaded {len(vectorstore.get()['ids'])} text chunks.")

--- Loading your SQL Database ---
Success! Loaded 438 text chunks.


In [5]:
# 4. Initialize Gemini for Generation
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [6]:
# 5. Define the Prompt & Chain
system_prompt = (
    "You are an expert SQL Architect. When asked to create tables:\n"
    "1. Always use the specific syntax found in the retrieved context.\n"
    "2. Standardize column names using underscores (e.g., user_id).\n"
    "3. Automatically include 'Primary Key' and 'Not Null' where logical.\n"
    "4. If multiple SQL dialects are present, default to the one used in the examples."
    "\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create the UI Elements
search_bar = widgets.Text(
    placeholder='Ask a SQL question...',
    description='Query:',
    layout=widgets.Layout(width='70%')
)
ask_button = widgets.Button(
    description='Ask AI',
    button_style='success', 
    icon='search'
)
output_area = widgets.Output()

# 2. Define the Action Function
def handle_query(b):
    with output_area:
        # Get the text and clear the previous answer
        user_query = search_bar.value
        if not user_query.strip():
            return
        
        clear_output() 
        print(f"Searching for: {user_query}...")
        
        try:
            # Run the existing RAG chain
            response = rag_chain.invoke({"input": user_query})
            
            print(f"\nAI Answer: {response['answer']}")
            
            # Display Sources
            pages = set([doc.metadata.get('page', 'N/A') for doc in response['context']])
            print(f"\nSources: Page(s) {', '.join(map(str, pages))}")
            
        except Exception as e:
            print(f"Error: {e}")

# 3. Connect the Button and Enter key to the function
ask_button.on_click(handle_query)
search_bar.on_submit(handle_query) 

# 4. Display the UI
print("--- SQL Knowledge Bot Interface ---")
display(widgets.HBox([search_bar, ask_button]), output_area)

--- SQL Knowledge Bot Interface ---


C:\Users\Swapn\AppData\Local\Temp\ipykernel_14940\947806751.py:43: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  search_bar.on_submit(handle_query)


Output()